# Transporte Público · RED Movilidad — Posiciones GPS Diarias

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Data-Observatory/datopia-notebooks/blob/main/transporte/red-movilidad/demo.ipynb)
[![Licencia](https://img.shields.io/badge/licencia-MIT-blue.svg)](../../LICENSE)
[![Dataset](https://img.shields.io/badge/dataset-RED%20Movilidad-green.svg)]()
[![Actualización](https://img.shields.io/badge/actualización-diaria-brightgreen.svg)]()
[![Python](https://img.shields.io/badge/python-3.10%2B-blue.svg)]()

---

## Descripción

Acceso al dataset de posiciones GPS diarias de la **RED Movilidad** (Región Metropolitana, Chile)
a través del **Datopia Lakehouse**. Los datos provienen de la API DTP Metropolitano y se
consolidan diariamente en formato **GeoParquet** particionado por fecha.

El acceso es directo vía S3 con credenciales temporales, sin descarga de archivos.

### Contenido

1. Configuración y autenticación
2. Exploración del dataset (catálogo, esquema, cobertura temporal)
3. Consulta semanal: registros, vehículos y servicios por día
4. Velocidad por hora del día (hora local)
5. Buses por día de semana (dic/ene vs mar/abr)
6. Actividad horaria — día representativo (hora local)
7. Mapa de posiciones espaciales

### Requisitos

Cuenta en el Datopia Lakehouse · Python 3.10+ · Las dependencias se instalan automáticamente

In [ ]:
# @title Instalación de dependencias
import importlib, subprocess, sys

for paquete in ["requests", "duckdb", "pandas", "matplotlib"]:
    if importlib.util.find_spec(paquete) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

import json, os, getpass, pathlib
import requests, duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

print("Dependencias listas.")

In [ ]:
# @title Configuración de credenciales
EN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
print(f"Entorno: {'Google Colab' if EN_COLAB else 'local'}")

URL_API = "https://ee98cnfz7e.execute-api.us-west-2.amazonaws.com/prod"

if EN_COLAB:
    EMAIL = input("Email: ").strip()
    PASSWORD = getpass.getpass("Contraseña: ")
else:
    ruta_cfg = pathlib.Path("../../config.json")
    if not ruta_cfg.exists():
        raise FileNotFoundError(
            f"Archivo de configuración no encontrado en {ruta_cfg.resolve()}.\n"
            "Copia config.example.json → config.json y completa tus credenciales."
        )
    cfg = json.loads(ruta_cfg.read_text())
    EMAIL = cfg.get("test_user", {}).get("email") or input("Email: ").strip()
    PASSWORD = cfg.get("test_user", {}).get("password") or getpass.getpass(
        "Contraseña: "
    )

print(f"API    : {URL_API}")
print(f"Usuario: {EMAIL}")

---
## 1. Autenticación y conexión a S3

In [ ]:
# Iniciar sesión y obtener credenciales temporales S3
resp_login = requests.post(
    f"{URL_API}/auth/login",
    json={"email": EMAIL, "password": PASSWORD},
    timeout=30,
)
resp_login.raise_for_status()
token = resp_login.json()["id_token"]

resp_s3 = requests.post(
    f"{URL_API}/auth/session/s3",
    headers={"Authorization": f"Bearer {token}"},
    timeout=30,
)
resp_s3.raise_for_status()
creds = resp_s3.json()

print("Sesión iniciada")
print(f"  Bucket : {creds['bucket']}")
print(f"  Región : {creds['region']}")
print(f"  Expira : {creds['expires_at']}")

In [ ]:
# Configurar DuckDB con credenciales S3
con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE SECRET s3 (
        TYPE          S3,
        KEY_ID        '{creds["access_key_id"]}',
        SECRET        '{creds["secret_access_key"]}',
        SESSION_TOKEN '{creds["session_token"]}',
        REGION        '{creds["region"]}'
    )
""")
# Extensión espacial para ST_X/ST_Y sobre columnas geometry (GeoParquet)
try:
    con.execute("LOAD spatial")
except Exception:
    con.execute("INSTALL spatial; LOAD spatial")
# Forzar UTC para que TIMESTAMP::TIMESTAMPTZ interprete correctamente los campos GPS
con.execute("SET TimeZone = 'UTC'")

BASE = (
    f"s3://{creds['bucket']}/categoria=transporte/pais=cl"
    f"/fuente=red-movilidad/tipo=posicion-diaria/version=v1"
)
print("DuckDB conectado a S3.")

---
## 2. Exploración del dataset

In [ ]:
# Metadatos del dataset (dataset.json en la raíz del path S3)
meta = (
    con.execute(f"SELECT * FROM read_json('{BASE}/dataset.json')")
    .df()
    .iloc[0]
    .to_dict()
)

print(meta["description"])
print()
print(f"Formato      : {meta['format']}")
print(f"Particiones  : {', '.join(meta['partition_keys'])}")
print(
    f"Geometría    : {meta['geometry']['geometry_type']} · {meta['geometry']['crs']} · {meta['geometry']['encoding']}"
)
print()

cols_list_of_dicts = [dict(col) for col in meta["columns"]]
cols = pd.DataFrame(cols_list_of_dicts)[["name", "type", "comment"]]
cols

In [ ]:
# Esquema de columnas
con.execute(f"""
    DESCRIBE SELECT * FROM read_parquet('{BASE}/year=2026/month=05/**/*.parquet') LIMIT 0
""").df()[["column_name", "column_type", "null"]]

In [ ]:
# Cobertura temporal disponible — lista archivos del lakehouse (no lee datos)
cobertura = con.execute(f"""
    SELECT
        regexp_extract(file, 'year=(\\d+)', 1)::INTEGER  AS año,
        regexp_extract(file, 'month=(\\d+)', 1)::INTEGER AS mes,
        COUNT(DISTINCT regexp_extract(file, 'day=(\\d+)', 1)::INTEGER) AS dias_disponibles
    FROM glob('{BASE}/**/*.parquet')
    GROUP BY año, mes
    ORDER BY año, mes
""").df()
print(f"Meses en el lakehouse: {len(cobertura)}")
cobertura

---
## 3. Consulta semanal

In [ ]:
# Ajusta año, mes y rango de días según la cobertura disponible arriba
AÑO = 2026
MES = 5
DIA_INICIO = 1
DIA_FIN = 7  # demo de una semana (~50-70 MB vs ~300 MB del mes completo)

# Rutas explícitas por día — DuckDB solo accede a esos 7 directorios
rutas_semana = [
    f"'{BASE}/year={AÑO}/month={MES:02d}/day={d:02d}/**/*.parquet'"
    for d in range(DIA_INICIO, DIA_FIN + 1)
]
ruta_semana = f"[{', '.join(rutas_semana)}]"

DIAS_ES = {
    "Monday": "Lunes",
    "Tuesday": "Martes",
    "Wednesday": "Miércoles",
    "Thursday": "Jueves",
    "Friday": "Viernes",
    "Saturday": "Sábado",
    "Sunday": "Domingo",
}

resumen_diario = con.execute(f"""
    SELECT
        day                                                       AS dia,
        DAYNAME(MAKE_DATE({AÑO}, {MES}, day::BIGINT))            AS dia_semana,
        COUNT(*)                                                  AS registros,
        COUNT(DISTINCT plate)                                     AS vehiculos,
        COUNT(DISTINCT service_name)                              AS servicios
    FROM read_parquet({ruta_semana}, hive_partitioning=true)
    GROUP BY dia
    ORDER BY dia
""").df()

resumen_diario["dia_semana"] = resumen_diario["dia_semana"].map(DIAS_ES)

print(
    f"Período: {AÑO}-{MES:02d} días {DIA_INICIO}–{DIA_FIN} — {len(resumen_diario)} días cargados"
)
resumen_diario

In [ ]:
# Muestra de registros GPS
con.execute(f"""
    SELECT timestamp_gps_utc, plate, service_name, speed, direction, way
    FROM read_parquet({ruta_semana}, hive_partitioning=true)
    LIMIT 10
""").df()

---
## 4. Velocidad por hora del día (hora local)

In [ ]:
# America/Santiago maneja DST automáticamente: UTC-3 (oct–mar, verano) · UTC-4 (abr–sep, invierno)
velocidad_hora = con.execute(f"""
    SELECT
        EXTRACT(HOUR FROM (timestamp_gps_utc::TIMESTAMPTZ AT TIME ZONE 'America/Santiago'))::INTEGER
            AS hora_local,
        ROUND(AVG(speed), 2)    AS velocidad_promedio_kmh,
        ROUND(MEDIAN(speed), 2) AS velocidad_mediana_kmh,
        COUNT(*)                AS registros
    FROM read_parquet({ruta_semana}, hive_partitioning=true)
    WHERE speed > 0
    GROUP BY hora_local
    ORDER BY hora_local
""").df()
velocidad_hora

---
## 5. Buses por día de semana

In [ ]:
# Vehículos únicos promedio por día de semana — dic/ene vs mar/abr
# Usa día UTC como proxy del día local (buses operan de día; frontera nocturna ~3-4 h no afecta el conteo)
MESES = [(2025, 12, "Dic"), (2026, 1, "Ene"), (2026, 3, "Mar"), (2026, 4, "Abr")]
DIAS_ES = {1: "Lun", 2: "Mar", 3: "Mié", 4: "Jue", 5: "Vie", 6: "Sáb", 7: "Dom"}
COLORES = {"Dic": "#1565C0", "Ene": "#42A5F5", "Mar": "#E65100", "Abr": "#FFA726"}

frames = []
for anio, mes, etiqueta in MESES:
    ruta_mes = f"'{BASE}/year={anio}/month={mes:02d}/**/*.parquet'"
    df = con.execute(f"""
        SELECT '{etiqueta}' AS mes, {mes} AS mes_num, day,
               ISODOW(MAKE_DATE({anio}, {mes}, day::BIGINT)) AS dow,
               COUNT(DISTINCT plate) AS vehiculos
        FROM read_parquet({ruta_mes}, hive_partitioning=true)
        GROUP BY day
    """).df()
    frames.append(df)

buses_dow = pd.concat(frames, ignore_index=True)
buses_dow["dia"] = buses_dow["dow"].map(DIAS_ES)
resumen_dow = (
    buses_dow.groupby(["mes", "mes_num", "dow", "dia"])["vehiculos"]
    .mean()
    .round(0)
    .reset_index()
    .sort_values(["mes_num", "dow"])
)

dias_orden = [DIAS_ES[i] for i in range(1, 8)]
x = list(range(len(dias_orden)))
ancho = 0.2

fig, ax = plt.subplots(figsize=(12, 5))
for i, (_, grp) in enumerate(resumen_dow.groupby("mes_num")):
    etiqueta = grp["mes"].iloc[0]
    vals = [
        grp.loc[grp["dia"] == d, "vehiculos"].values[0] if d in grp["dia"].values else 0
        for d in dias_orden
    ]
    ax.bar(
        [xi + (i - 1.5) * ancho for xi in x],
        vals,
        ancho,
        label=etiqueta,
        color=COLORES[etiqueta],
        edgecolor="white",
    )

ax.set_xticks(x)
ax.set_xticklabels(dias_orden)
ax.set_title("Vehículos únicos promedio por día de semana", fontweight="bold")
ax.set_ylabel("Vehículos")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.legend(title="Mes")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Actividad horaria — día representativo (hora local)

In [ ]:
# 16 de marzo 2026 en hora local (CLT = UTC-3 en verano)
# Día local 2026-03-16 = UTC 03:00 día 16 → UTC 02:59 día 17
# → cargar particiones UTC day=16 y day=17, filtrar por fecha local

ruta_marzo_rep = (
    f"['{BASE}/year=2026/month=03/day=16/**/*.parquet',"
    f" '{BASE}/year=2026/month=03/day=17/**/*.parquet']"
)

hora_marzo = con.execute(f"""
    SELECT
        EXTRACT(HOUR FROM (timestamp_gps_utc::TIMESTAMPTZ AT TIME ZONE 'America/Santiago'))::INTEGER
            AS hora_local,
        ROUND(AVG(speed), 2)  AS velocidad_promedio_kmh,
        COUNT(DISTINCT plate) AS vehiculos,
        COUNT(*)              AS registros
    FROM read_parquet({ruta_marzo_rep}, hive_partitioning=true)
    WHERE
        (timestamp_gps_utc::TIMESTAMPTZ AT TIME ZONE 'America/Santiago')::DATE = DATE '2026-03-16'
        AND speed > 0
    GROUP BY hora_local
    ORDER BY hora_local
""").df()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Lunes 16 de marzo 2026 — hora local (CLT, UTC-3)", fontweight="bold")

ax = axes[0]
ax.plot(
    hora_marzo["hora_local"],
    hora_marzo["vehiculos"],
    marker="o",
    color="#1976D2",
    linewidth=2,
)
ax.set_title("Vehículos activos por hora")
ax.set_xlabel("Hora local")
ax.set_ylabel("Vehículos únicos")
ax.set_xticks(range(0, 24, 2))
ax.grid(alpha=0.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))

ax = axes[1]
ax.plot(
    hora_marzo["hora_local"],
    hora_marzo["velocidad_promedio_kmh"],
    marker="o",
    color="#E53935",
    linewidth=2,
)
ax.set_title("Velocidad promedio por hora")
ax.set_xlabel("Hora local")
ax.set_ylabel("Velocidad (km/h)")
ax.set_xticks(range(0, 24, 2))
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 7. Mapa de posiciones espaciales

In [ ]:
# Posiciones GPS en hora punta — 1 mayo 2026 (invierno, CLT = UTC-4)
# Rush mañana: 07:30–08:00 CLT → 11:30–12:00 UTC
# Rush tarde : 17:30–18:00 CLT → 21:30–22:00 UTC

ruta_mapa = f"'{BASE}/year=2026/month=05/day=01/**/*.parquet'"

ventanas = [
    ("Rush mañana (07:30–08:00 CLT)", "2026-05-01 11:30:00", "2026-05-01 12:00:00"),
    ("Rush tarde  (17:30–18:00 CLT)", "2026-05-01 21:30:00", "2026-05-01 22:00:00"),
]

fig, axes = plt.subplots(1, 2, figsize=(16, 10))
fig.suptitle(
    "RED Movilidad — posiciones GPS · 1 mayo 2026", fontweight="bold", fontsize=14
)

for ax, (titulo, t_ini, t_fin) in zip(axes, ventanas):
    pos = con.execute(f"""
        SELECT ST_X(geometry) AS lon, ST_Y(geometry) AS lat, speed
        FROM read_parquet({ruta_mapa}, hive_partitioning=true)
        WHERE timestamp_gps_utc BETWEEN '{t_ini}' AND '{t_fin}'
          AND geometry IS NOT NULL
    """).df()
    sc = ax.scatter(
        pos["lon"],
        pos["lat"],
        c=pos["speed"],
        cmap="RdYlGn",
        s=1,
        alpha=0.5,
        vmin=0,
        vmax=60,
    )
    plt.colorbar(sc, ax=ax, label="km/h", shrink=0.7)
    ax.set_title(f"{titulo}\n{len(pos):,} registros", fontweight="bold")
    ax.set_xlabel("Longitud")
    ax.set_ylabel("Latitud")
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()

---
## Próximos pasos

- **Análisis espacial:** usar la columna `geometry` (WKB Point EPSG:4326) con `geopandas` para mapas de trayectorias.
- **Múltiples meses:** ajustar `AÑO`/`MES` o iterar sobre rangos para análisis temporales largos.
- **Por servicio:** filtrar por `service_name` para analizar una línea específica (ej. `T201`, `B01`).
- **Más datasets:** explorar otros conjuntos disponibles en [datopia-notebooks](https://github.com/Data-Observatory/datopia-notebooks).

---

*Datos provistos por [Data Observatory](https://dataobservatory.net) · Fuente: DTP Metropolitano (DTPM)*